In [1]:
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from imblearn.over_sampling import RandomOverSampler

In [ ]:
# 0) Paths and settings

DATA_PATH = "../data/cleaned/cicids2017_cleaned_nids.csv"
OUTPUT_DIR = "../data/processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TEST_SIZE = 0.20
RANDOM_STATE = 42
OUTLIER_CONTAMINATION = 0.02
CORR_THRESHOLD = 0.95

# Classes with fewer rows than this are merged
MIN_CLASS_COUNT_FOR_SPLIT = 2

# Cap oversampling target to avoid huge memory usage
MAX_OVERSAMPLE_TARGET = 100000

In [ ]:
# 1) Load data

data = pd.read_csv(DATA_PATH)
data.columns = data.columns.str.strip()

print("Loaded dataset shape:", data.shape)

if "Label" not in data.columns:
    raise ValueError(f"Column 'Label' not found. Available columns: {list(data.columns)}")

X = data.drop("Label", axis=1).copy()
y = data["Label"].astype(str).str.strip().copy()

label_cleanup = {
    "Web Attack \ufffd XSS": "Web Attack - XSS",
    "Web Attack \ufffd Brute Force": "Web Attack - Brute Force",
    "Web Attack \ufffd Sql Injection": "Web Attack - Sql Injection",
    "Web Attack – XSS": "Web Attack - XSS",
    "Web Attack – Brute Force": "Web Attack - Brute Force",
    "Web Attack – Sql Injection": "Web Attack - Sql Injection",
}
y = y.replace(label_cleanup)

print("Intrusion categories found:")
print(sorted(y.unique()))
print("\nClass distribution before rare-class handling:")
print(y.value_counts())

Loaded dataset shape: (2522009, 71)
Intrusion categories found:
['BENIGN', 'Bot', 'DDoS', 'DoS GoldenEye', 'DoS Hulk', 'DoS Slowhttptest', 'DoS slowloris', 'FTP-Patator', 'Heartbleed', 'Infiltration', 'PortScan', 'SSH-Patator', 'Web Attack - Brute Force', 'Web Attack - Sql Injection', 'Web Attack - XSS']

Class distribution before rare-class handling:
Label
BENIGN                        2096134
DoS Hulk                       172846
DDoS                           128016
PortScan                        90819
DoS GoldenEye                   10286
FTP-Patator                      5933
DoS slowloris                    5385
DoS Slowhttptest                 5228
SSH-Patator                      3219
Bot                              1953
Web Attack - Brute Force         1470
Web Attack - XSS                  652
Infiltration                       36
Web Attack - Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64


In [ ]:
# 2) Encode feature-side categoricals only

categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()

if categorical_cols:
    print("Categorical feature columns detected:", categorical_cols)
    X = pd.get_dummies(X, columns=categorical_cols, drop_first=False)
else:
    print("No categorical feature columns detected.")

print("Feature matrix shape after categorical encoding:", X.shape)

No categorical feature columns detected.
Feature matrix shape after categorical encoding: (2522009, 70)


In [ ]:
# 3) Remove zero-variance columns

zero_var_cols = [col for col in X.columns if X[col].nunique(dropna=False) <= 1]

if zero_var_cols:
    X = X.drop(columns=zero_var_cols)

print("Zero variance columns removed:", len(zero_var_cols))
print("Feature matrix shape after zero-variance removal:", X.shape)

Zero variance columns removed: 0
Feature matrix shape after zero-variance removal: (2522009, 70)


In [ ]:
# 4) Handle rare classes BEFORE split

class_counts = y.value_counts()
rare_classes = class_counts[class_counts < MIN_CLASS_COUNT_FOR_SPLIT].index.tolist()

if rare_classes:
    print("Rare classes detected and merged into 'Other_Attack':", rare_classes)
    y = y.replace(rare_classes, "Other_Attack")
else:
    print("No rare classes below minimum threshold.")

print("\nClass distribution after rare-class handling:")
print(y.value_counts())

No rare classes below minimum threshold.

Class distribution after rare-class handling:
Label
BENIGN                        2096134
DoS Hulk                       172846
DDoS                           128016
PortScan                        90819
DoS GoldenEye                   10286
FTP-Patator                      5933
DoS slowloris                    5385
DoS Slowhttptest                 5228
SSH-Patator                      3219
Bot                              1953
Web Attack - Brute Force         1470
Web Attack - XSS                  652
Infiltration                       36
Web Attack - Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64


In [ ]:
# 5) Train / test split

X_train_df, X_test_df, y_train_labels, y_test_labels = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

X_train_df = X_train_df.reset_index(drop=True)
X_test_df = X_test_df.reset_index(drop=True)
y_train_labels = y_train_labels.reset_index(drop=True)
y_test_labels = y_test_labels.reset_index(drop=True)

print("Train shape before train-only preprocessing:", X_train_df.shape)
print("Test shape:", X_test_df.shape)

print("\nTraining class distribution:")
print(y_train_labels.value_counts())

print("\nTest class distribution:")
print(y_test_labels.value_counts())

Train shape before train-only preprocessing: (2017607, 70)
Test shape: (504402, 70)

Training class distribution:
Label
BENIGN                        1676907
DoS Hulk                       138277
DDoS                           102413
PortScan                        72655
DoS GoldenEye                    8229
FTP-Patator                      4746
DoS slowloris                    4308
DoS Slowhttptest                 4182
SSH-Patator                      2575
Bot                              1562
Web Attack - Brute Force         1176
Web Attack - XSS                  522
Infiltration                       29
Web Attack - Sql Injection         17
Heartbleed                          9
Name: count, dtype: int64

Test class distribution:
Label
BENIGN                        419227
DoS Hulk                       34569
DDoS                           25603
PortScan                       18164
DoS GoldenEye                   2057
FTP-Patator                     1187
DoS slowloris                 

In [ ]:
# 6) Handle inf / missing values using TRAIN statistics only

X_train_df = X_train_df.replace([np.inf, -np.inf], np.nan)
X_test_df = X_test_df.replace([np.inf, -np.inf], np.nan)

train_means = X_train_df.mean(numeric_only=True)

X_train_df = X_train_df.fillna(train_means)
X_test_df = X_test_df.fillna(train_means)

print("Missing / infinite values handled using training means.")
print("Remaining NaNs in X_train:", int(X_train_df.isna().sum().sum()))
print("Remaining NaNs in X_test :", int(X_test_df.isna().sum().sum()))

Missing / infinite values handled using training means.
Remaining NaNs in X_train: 0
Remaining NaNs in X_test : 0


In [ ]:
# 7) Outlier removal ONLY on training data

iso = IsolationForest(
    contamination=OUTLIER_CONTAMINATION,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

train_pred = iso.fit_predict(X_train_df)
train_mask = train_pred != -1

X_train_df = X_train_df.loc[train_mask].reset_index(drop=True)
y_train_labels = y_train_labels.loc[train_mask].reset_index(drop=True)

print("Train shape after outlier removal:", X_train_df.shape)
print("Training class distribution after outlier removal:")
print(y_train_labels.value_counts())

Train shape after outlier removal: (1977254, 70)
Training class distribution after outlier removal:
Label
BENIGN                        1643718
DoS Hulk                       134672
DDoS                           101569
PortScan                        72626
DoS GoldenEye                    8197
FTP-Patator                      4746
DoS Slowhttptest                 3834
SSH-Patator                      2575
DoS slowloris                    2141
Bot                              1529
Web Attack - Brute Force         1112
Web Attack - XSS                  509
Web Attack - Sql Injection         17
Infiltration                        9
Name: count, dtype: int64


In [ ]:
# 8) Remove highly correlated features learned from TRAIN only

corr_matrix = X_train_df.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
corr_drop = [col for col in upper.columns if any(upper[col] > CORR_THRESHOLD)]

if corr_drop:
    X_train_df = X_train_df.drop(columns=corr_drop)
    X_test_df = X_test_df.drop(columns=corr_drop, errors="ignore")

print("Highly correlated columns removed:", len(corr_drop))
print("Train shape after correlation removal:", X_train_df.shape)
print("Test shape after correlation removal:", X_test_df.shape)

Highly correlated columns removed: 23
Train shape after correlation removal: (1977254, 47)
Test shape after correlation removal: (504402, 47)


In [ ]:
# 9) Scale using TRAIN only

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_df).astype(np.float32)
X_test_scaled = scaler.transform(X_test_df).astype(np.float32)

print("Scaled train shape:", X_train_scaled.shape)
print("Scaled test shape :", X_test_scaled.shape)
print("Scaled dtype      :", X_train_scaled.dtype)

Scaled train shape: (1977254, 47)
Scaled test shape : (504402, 47)
Scaled dtype      : float32


In [ ]:
# 10) Multiclass oversampling ONLY on training data
#     with a cap to avoid huge memory usage

class_counts_before = pd.Series(y_train_labels).value_counts().sort_index()

print("Training class distribution before balancing:")
print(class_counts_before)

majority_count = int(class_counts_before.max())
target_count = int(min(majority_count, MAX_OVERSAMPLE_TARGET))

sampling_strategy = {
    cls: target_count
    for cls, count in class_counts_before.items()
    if count < target_count
}

print("\nOversampling target per minority class:", target_count)
print("Number of classes to oversample:", len(sampling_strategy))

if sampling_strategy:
    ros = RandomOverSampler(
        sampling_strategy=sampling_strategy,
        random_state=RANDOM_STATE
    )
    X_train_resampled, y_train_resampled = ros.fit_resample(X_train_scaled, y_train_labels)
else:
    X_train_resampled = X_train_scaled
    y_train_resampled = y_train_labels.copy()

X_train_resampled = np.asarray(X_train_resampled, dtype=np.float32)
y_train_resampled = pd.Series(y_train_resampled).reset_index(drop=True)

print("\nTraining class distribution after balancing:")
print(y_train_resampled.value_counts().sort_index())
print("Resampled X_train shape:", X_train_resampled.shape)

Training class distribution before balancing:
Label
BENIGN                        1643718
Bot                              1529
DDoS                           101569
DoS GoldenEye                    8197
DoS Hulk                       134672
DoS Slowhttptest                 3834
DoS slowloris                    2141
FTP-Patator                      4746
Infiltration                        9
PortScan                        72626
SSH-Patator                      2575
Web Attack - Brute Force         1112
Web Attack - Sql Injection         17
Web Attack - XSS                  509
Name: count, dtype: int64

Oversampling target per minority class: 100000
Number of classes to oversample: 11

Training class distribution after balancing:
Label
BENIGN                        1643718
Bot                            100000
DDoS                           101569
DoS GoldenEye                  100000
DoS Hulk                       134672
DoS Slowhttptest               100000
DoS slowloris             

In [ ]:
# 11) Encode labels for model training
#     (keep category names recoverable later)

label_encoder = LabelEncoder()

all_labels = pd.concat(
    [
        pd.Series(y_train_resampled),
        pd.Series(y_test_labels)
    ],
    axis=0
)

label_encoder.fit(all_labels)

y_train = label_encoder.transform(y_train_resampled)
y_test = label_encoder.transform(y_test_labels)

X_train = X_train_resampled.astype(np.float32)
X_test = X_test_scaled.astype(np.float32)

print("Final X_train:", X_train.shape)
print("Final X_test :", X_test.shape)
print("Final X_train dtype:", X_train.dtype)

print("\nEncoded class mapping:")
for idx, cls in enumerate(label_encoder.classes_):
    print(f"{idx} -> {cls}")

Final X_train: (2979959, 47)
Final X_test : (504402, 47)
Final X_train dtype: float32

Encoded class mapping:
0 -> BENIGN
1 -> Bot
2 -> DDoS
3 -> DoS GoldenEye
4 -> DoS Hulk
5 -> DoS Slowhttptest
6 -> DoS slowloris
7 -> FTP-Patator
8 -> Heartbleed
9 -> Infiltration
10 -> PortScan
11 -> SSH-Patator
12 -> Web Attack - Brute Force
13 -> Web Attack - Sql Injection
14 -> Web Attack - XSS


In [ ]:
# 12) Save outputs

np.save(os.path.join(OUTPUT_DIR, "X_train.npy"), X_train)
np.save(os.path.join(OUTPUT_DIR, "X_test.npy"), X_test)
np.save(os.path.join(OUTPUT_DIR, "y_train.npy"), y_train)
np.save(os.path.join(OUTPUT_DIR, "y_test.npy"), y_test)

pd.Series(y_train_resampled, name="Label").to_csv(
    os.path.join(OUTPUT_DIR, "y_train_labels.csv"), index=False
)
pd.Series(y_test_labels, name="Label").to_csv(
    os.path.join(OUTPUT_DIR, "y_test_labels.csv"), index=False
)

artifacts = {
    "scaler": scaler,
    "label_encoder": label_encoder,
    "selected_columns": list(X_train_df.columns),
    "corr_drop": corr_drop,
    "zero_var_cols": zero_var_cols,
    "train_means": train_means,
    "expected_input_columns": list(X.columns)
}

joblib.dump(artifacts, "../data/processed/preprocessing_artifacts.joblib")

print("Preprocessing completed and files saved successfully.")

Preprocessing completed and files saved successfully.


In [ ]:
# 13) Quick sanity check

print("Saved files:")
for filename in [
    "X_train.npy",
    "X_test.npy",
    "y_train.npy",
    "y_test.npy",
    "y_train_labels.csv",
    "y_test_labels.csv",
    "preprocessing_artifacts.joblib",
]:
    full_path = os.path.join(OUTPUT_DIR, filename)
    print(f"{filename}: {os.path.exists(full_path)}")

Saved files:
X_train.npy: True
X_test.npy: True
y_train.npy: True
y_test.npy: True
y_train_labels.csv: True
y_test_labels.csv: True
preprocessing_artifacts.joblib: True
